In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,lastFalsePositive,indicator
0,5629499555000444,2025-06-13T14:11:03Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T14:05:11Z,3.0,66,INC9102471,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.103.172.199
1,5629499563382886,2025-08-14T12:19:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:48:18Z,3.0,74,INC9193873,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,150.107.38.251
2,5629499546480685,2025-06-02T19:00:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:38:41Z,3.0,64,TASK0882701 / RITM0585661,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.26.111.94
3,6755399443015053,2025-03-14T11:55:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:38:36Z,4.0,54,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.47
4,6755399443015044,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:38:36Z,4.0,52,CMS received a Volumetrics alert regarding mul...,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,4503719,2024-01-23T19:14:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:43Z,3.0,80,A phishing email with a .htm file containing a...,...,"[{'id': 238537, 'name': 'fake voicemail', 'las...","[{'id': 308984, 'dateAdded': '2024-01-23T19:14...",NaN,NaN,NaN,https://aka.ms/LearnAboutSenderIdentification,NaN,aka.ms/learnaboutsenderidentification,NaN,aka.ms/learnaboutsenderidentification
1171,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,NaN,geo.netsupportsoftware.com/location/loca.asp
1172,4598575,2024-05-10T15:06:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-05-10T15:14:45Z,3.0,70,"On April 2, 2024, a VA user successfully conne...",...,"[{'id': 38969, 'name': 'Stage Capabilities: SE...","[{'id': 335586, 'dateAdded': '2024-05-10T14:40...",NaN,NaN,NaN,NaN,NaN,172.67.156.141,NaN,172.67.156.141
1173,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,https://google,NaN,google,NaN,google


In [3]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Fields to return from GET /v3/indicators/{indicatorId or indicator}
detail_fields = [
    "ownerName", "type", "summary", "dateAdded", "lastModified", "active",
    "confidence", "rating", "threatAssessScore", "observations",
    "tags", "attributes", "securityLabels", "associatedGroups"
]

list_of_indicators = ['207.167.64.24'] 

# Build the keys to fetch
if "observed_src" in globals() and isinstance(observed_src, pd.DataFrame) and not observed_src.empty:
    if "id" in observed_src.columns:
        indicator_keys = (
            observed_src["id"].dropna().astype(str).str.strip().tolist()
        )
    elif "summary" in observed_src.columns:
        indicator_keys = (
            observed_src["summary"].dropna().astype(str).str.strip().tolist()
        )
    else:
        indicator_keys = list_of_indicators
else:
    indicator_keys = list_of_indicators

# De-duplicate while preserving order
seen = set()
ordered_keys = []
for k in indicator_keys:
    if k and k not in seen:
        seen.add(k)
        ordered_keys.append(k)

final_results = []

# Query indicators (single-resource endpoint only)
for key in ordered_keys:
    display(f"Querying indicator: {key}")
    try:
        # URL-encode the path segment (works for both numeric IDs and summary strings)
        path_key = urllib.parse.quote(str(key), safe="")
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{path_key}?fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            result = response.json() or {}
            final_results.append(result)
    except Exception as e:
        pass

# Normalize results
normalized_data = []
for result in final_results:
    data_item = result.get('data', {})
    # v3 single GET usually returns a dict; occasionally data['indicator'] may hold it
    if isinstance(data_item, dict):
        record = data_item.get('indicator', data_item)
        if isinstance(record, dict) and 'summary' in record:
            normalized_data.append(record)
    elif isinstance(data_item, list):
        # Very rare shape; keep parity with earlier normalizer
        for item in data_item:
            if isinstance(item, dict) and 'summary' in item:
                normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    # Derive a simple 'indicator' token from summary (e.g., first token)
    if 'summary' in observed_src.columns:
        observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()

    # Drop dupes on 'indicator' if it exists
    if 'indicator' in observed_src.columns:
        observed_src.drop_duplicates(subset='indicator', inplace=True)

    # Filter to lastObserved >= start, if lastObserved exists
    if 'lastObserved' in observed_src.columns:
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
        #observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying indicator: 5629499555000444'

'Querying indicator: 5629499563382886'

'Querying indicator: 5629499546480685'

'Querying indicator: 6755399443015053'

'Querying indicator: 6755399443015044'

...

'Querying indicator: 4324196'

'Querying indicator: 4598575'

'Querying indicator: 4442727'

'Querying indicator: 4481963'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,5629499555000444,2025-06-13T14:11:03Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T14:05:11Z,3.0,66,INC9102471,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 23575, 'name': 'CDC Splunk API', 'last...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.103.172.199
1,5629499563382886,2025-08-14T12:19:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:48:18Z,3.0,74,INC9193873,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 749, 'name': 'malicious', 'lastUsed': ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,150.107.38.251
2,5629499546480685,2025-06-02T19:00:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:38:41Z,3.0,64,TASK0882701 / RITM0585661,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 749, 'name': 'malicious', 'lastUsed': ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.26.111.94
3,6755399443015053,2025-03-14T11:55:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:38:36Z,4.0,54,CMS received a Volumetrics alert regarding mul...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 748, 'name': 'malware', 'lastUsed': '2...","[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.47
4,6755399443015044,2025-03-14T11:55:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T13:38:36Z,4.0,52,CMS received a Volumetrics alert regarding mul...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 748, 'name': 'malware', 'lastUsed': '2...","[{'id': 6755399443002242, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,91.196.152.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,4503719,2024-01-23T19:14:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:43Z,3.0,80,A phishing email with a .htm file containing a...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1740, 'name': 'TLP:AMBER', 'lastUsed':...","[{'id': 308984, 'dateAdded': '2024-01-23T19:14...",NaN,NaN,NaN,https://aka.ms/LearnAboutSenderIdentification,NaN,aka.ms/learnaboutsenderidentification,aka.ms/learnaboutsenderidentification
1168,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 4304, 'name': 'NETSUPPORT RAT', 'lastU...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1169,4598575,2024-05-10T15:06:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-05-10T15:14:45Z,3.0,70,"On April 2, 2024, a VA user successfully conne...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 34031, 'name': 'Malware Family: GOOTLO...","[{'id': 335586, 'dateAdded': '2024-05-10T14:40...",NaN,NaN,NaN,NaN,NaN,172.67.156.141,172.67.156.141
1170,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 23576, 'name': 'Observed', 'lastUsed':...",NaN,NaN,NaN,NaN,https://google,NaN,google,google


In [4]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,14.103.172.199,"[CDC Splunk API, Observed, HHS Splunk API, IHS..."
1,150.107.38.251,"[malicious, CDC Splunk API, Observed, IHS Splu..."
2,118.26.111.94,"[malicious, CDC Splunk API, Observed, HHS Splu..."
3,91.196.152.47,"[malware, Observed, HHS Splunk API, IHS Splunk..."
4,91.196.152.45,"[malware, Observed, HHS Splunk API, IHS Splunk..."
...,...,...
1167,aka.ms/learnaboutsenderidentification,"[TLP:AMBER, Observed, business email compromis..."
1168,geo.netsupportsoftware.com/location/loca.asp,"[NETSUPPORT RAT, malicious IP and Domain, fs-i..."
1169,172.67.156.141,"[Malware Family: GOOTLOADER, storm-0494, Drive..."
1170,google,"[Observed, HRSA Splunk API, OS Splunk API]"


In [5]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,1
1,DDoS,126
2,Spam,0
3,Phishing,25
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,11
7,Data Theft,6
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [46]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_24276\1319479530.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250827.csv']"

'Loaded data from 1 files.'

In [6]:
import pandas as pd

df = observed_src.copy()

#df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

#groups_exploded = (
#    df[['indicator', 'associatedGroups.data']]
#      .explode('associatedGroups.data')
#      .dropna(subset=['associatedGroups.data'])
#)

#group_norm = pd.json_normalize(
#    groups_exploded['associatedGroups.data']
#).rename(columns={'id':'group_id','name':'group_name'})

#groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

#group_agg = (
#    groups_exploded
#      .drop_duplicates(subset=['indicator','group_id','group_name'])
#      .groupby('indicator', as_index=False)
#      .agg({
#          'group_id':   lambda ids: list(ids),
#          'group_name': lambda names: list(names)
#      })
#      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
#)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
#for col in ['partners','group_ids','group_names'] + tag_fields:
#    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

# Only apply to columns that exist in agg_df to avoid KeyError
for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_id,tag_name,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners
0,1.4.195.14,6755399460007413,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-24T19:49:50Z,4.0,62,...,"11, 12, 98, 23576, 23769, 505200, 1455870, 146...","Hacktivism, DDoS, Iran, Observed, VA CSOC CTS ...","2025-07-31T19:26:57Z, 2025-07-31T19:14:54Z, 20...",,,,,,,
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-24T17:08:30Z,3.0,63,...,"749, 23576, 23577, 23586, 23630, 23667, 30479,...","malicious, Observed, HHS Splunk API, IHS Splun...","2025-09-23T23:44:52Z, 2025-09-24T21:52:19Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"HHS, IHS, NIH, HRSA, CMS, FDA, OS, DHA"
2,103.133.60.12,6755399460008266,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T11:03:58Z,1.0,62,...,"11, 12, 98, 23576, 23577, 23769, 35057, 471298...","Hacktivism, DDoS, Iran, Observed, HHS Splunk A...","2025-07-31T19:26:57Z, 2025-07-31T19:14:54Z, 20...",,,,,,,"HHS, FDA, DHA"
3,103.161.153.178,6755399460007422,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-20T18:22:23Z,4.0,62,...,"11, 12, 98, 23576, 23577, 23630, 23769, 30479,...","Hacktivism, DDoS, Iran, Observed, HHS Splunk A...","2025-07-31T19:26:57Z, 2025-07-31T19:14:54Z, 20...",,,,,,,"HHS, NIH, CMS, DHA"
4,103.163.103.50,6755399460007966,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T05:48:37Z,4.0,62,...,"11, 12, 98, 23576, 35057, 471298, 505200, 1455...","Hacktivism, DDoS, Iran, Observed, FDA Splunk A...","2025-07-31T19:26:57Z, 2025-07-31T19:14:54Z, 20...",,,,,,,"FDA, DHA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-24T22:19:09Z,3.0,74,...,"23576, 23586, 23630, 23769, 30479, 35752, 4615...","Observed, IHS Splunk API, NIH Splunk API, VA C...","2025-09-24T21:52:19Z, 2025-09-24T17:26:47Z, 20...",,,,,,,"IHS, NIH, CMS, DHA"
1168,www.prosinthecity.com,5263308,2025-01-22T18:43:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-25T11:24:03Z,4.0,57,...,"23576, 23630, 28007, 30479, 35760, 37143, 4673...","Observed, NIH Splunk API, SocGholish, CMS Splu...","2025-09-24T21:52:19Z, 2025-09-24T21:51:59Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"NIH, CMS, OS, DHA"
1169,www.shorturl.at/,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90,...,"26, 1502, 23575, 23576, 35057, 390199","Credential Harvesting, malspam, CDC Splunk API...","2025-09-19T00:00:00Z, 2025-09-19T00:00:00Z, 20...",,,,,,,"CDC, FDA"
1170,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-24T14:19:01Z,4.0,42,...,"5245, 23509, 23576, 23586, 23630, 23667, 23769...","VA_CTI_Vetted, malicious domain, Observed, IHS...","2025-09-17T02:09:27Z, 2025-09-23T05:48:07Z, 20...","Vetted IOCs, This tag is used specifically for...",,,,,,"IHS, NIH, HRSA, CMS, FDA, OS, DHA"


In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

display(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        display(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        display("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    display(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


'Enriching 1172 indicators with Shodan & VirusTotalV3...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL 172.67.156.141 cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL accentcare.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL aka.ms/learnaboutsenderidentification cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL aka.ms/o0ukef cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host api-docs.deepseek.com cannot be enriched with Shodan because the indicator type isn\'t supported.","st

'Successfully enriched and merged 1084 indicators.'

'88 indicators failed to enrich.'

In [8]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.4.195.14,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-24T19:49:50Z,4.0,62,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Khueang Nai],[Thailand],"[nt-isp.net, totinternet.net]","[node-d8u.pool-1-4.dynamic.totinternet.net, no...",[TOT Public Company Limited],"[{'transport': 'tcp', 'port': 88, 'product': '...",[TOT Public Company Limited],None,None,1.0
1,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-24T17:08:30Z,3.0,63,INC9067814,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,7.0
2,103.133.60.12,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T11:03:58Z,1.0,62,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Terbanggi Besar],[Indonesia],None,None,[PT Tunas Link Indonesia],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Tunas Link Indonesia],None,None,2.0
3,103.161.153.178,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-20T18:22:23Z,4.0,62,"Details\nIn late June 2025, Iran-aligned hackt...",...,None,None,None,None,None,None,None,None,None,0.0
4,103.163.103.50,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-25T05:48:37Z,4.0,62,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Sragen],[Indonesia],[mamura.net.id],[50.103-163-103.mamura.net.id],[PT Mamura Inter Media],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Mamura Inter Media],[vpn],None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-24T22:19:09Z,3.0,74,The following list of urls and domains are cur...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1168,www.prosinthecity.com,2025-01-22T18:43:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-25T11:24:03Z,4.0,57,socgholish Fake Update C2 malware\nA user was ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1169,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90,ACD R&F processed a malspam campaign with a Ne...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1170,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-24T14:19:01Z,4.0,42,The Department of Veterans Affairs (VA) receiv...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Convert DataFrame to dictionary records
records = recent_tags.to_dict(orient="records")

# Insert records into MongoDB
result = collection.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} documents into enriched_observation_Data.")

client.close()


Inserted 1172 documents into enriched_observation_Data.


In [10]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Pull data back from MongoDB
docs = list(collection.find())
recent_tags = pd.DataFrame(docs)
print(f"Pulled {len(recent_tags)} documents from enriched_observation_Data.")


Pulled 5117 documents from enriched_observation_Data.


In [12]:
#recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,68af160dc60b14cf6da41a47,1.192.18.4,2025-05-14T17:47:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T07:41:56Z,0.0,3.0,...,None,None,None,None,None,None,None,None,None,4.0
1,68af160dc60b14cf6da41a48,100.27.42.247,2025-08-11T15:16:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T07:42:04Z,0.0,3.0,...,None,None,None,None,None,None,None,None,None,0.0
2,68af160dc60b14cf6da41a49,101.89.148.7,2025-08-09T18:13:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T02:03:09Z,0.0,3.0,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 9999, 'data': 'H...",[CHINANET SHANGHAI PROVINCE NETWORK],None,None,11.0
3,68af160dc60b14cf6da41a4a,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-26T08:07:02Z,0.0,3.0,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],None,None,8.0
4,68af160dc60b14cf6da41a4b,102.0.5.152,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T13:37:36Z,0.0,4.0,...,[Nairobi],[Kenya],[airtelkenya.com],[152-5-0-102.r.airtelkenya.com],[Airtel Networks Kenya Limited],"[{'transport': 'udp', 'port': 161, 'product': ...",[Allocated to Airtel Kenya Mobile Broadband an...,[vpn],None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5112,68d553716441060dfec9758b,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-24T22:19:09Z,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5113,68d553716441060dfec9758c,www.prosinthecity.com,2025-01-22T18:43:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-25T11:24:03Z,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5114,68d553716441060dfec9758d,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5115,68d553716441060dfec9758e,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-24T14:19:01Z,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [13]:
# Count how many indicators are associated with a unique enrich_domains value

# Drop rows where enrich_domains is missing or NaN
domains_df = exploded.dropna(subset=['enrich_domains'])

# If enrich_domains is a list, flatten to individual domain rows
def flatten_domains(row):
    val = row['enrich_domains']
    if isinstance(val, list):
        return [(row['indicator'], d) for d in val if pd.notna(d)]
    elif pd.notna(val):
        return [(row['indicator'], val)]
    return []

flat = []
for _, row in domains_df.iterrows():
    flat.extend(flatten_domains(row))

flat_df = pd.DataFrame(flat, columns=['indicator', 'domain'])

# Count unique indicators per domain
domain_counts = (
    flat_df.groupby('domain')['indicator']
    .nunique()
    .reset_index()
    .rename(columns={'indicator': 'indicator_count'})
    .sort_values('indicator_count', ascending=False)
)

display(domain_counts)

KeyError: ['enrich_domains']

In [14]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_30924\1921769075.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250925.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250924.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250923.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250922.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250921.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250920.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250919.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250918.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250917.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250916.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250915.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250914.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [15]:
# Aggregate observed_data_df by 'indicator', counting unique obs_date occurrences
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)

agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
# Map obs_days_count from agg_by_indicator to recent_tags by indicator, rename to obs_count
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)

,indicator,obs_days_count
1,1.192.18.4,18
3,1.4.195.14,2
8,100.27.42.247,18
16,101.89.148.7,45
17,101.89.174.236,128
...,...,...
5086,www.deepseek.com.cdn.cloudflare.net,176
5117,www.prosinthecity.com,13
5119,www.shorturl.at/,181
5122,www.sthda.com,247


In [16]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

# Try to match on 'enrich_org' instead of 'enrich_hostNames'
if 'enrich_org' in recent_tags.columns:
    global matched_results
    exploded = recent_tags.explode('enrich_org')

    mask = exploded['enrich_org'].astype(str).str.match(four_label_word_prefix, na=False)

    matched = recent_tags.loc[exploded[mask].index.unique()]
    matched_results = matched[['indicator','enrich_org']]
else:
    matched = pd.DataFrame(columns=recent_tags.columns)
    matched_results = None


display(matched_results)


,indicator,enrich_org


In [17]:
def has_false_positive(tag_str):
    if not isinstance(tag_str, str):
        return False
    tags = [t.strip().lower() for t in tag_str.split(",")]
    return any("false positive" in t for t in tags)

false_positive_indicators = recent_tags[
    recent_tags['tag_name'].apply(has_false_positive)
]

display(false_positive_indicators[['indicator', 'tag_name']])


,indicator,tag_name


In [ ]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights ─────────────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 2.00,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
}

# ── Continuity Map ──────────────────────────────
CONTINUITY_MAP = {
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}

BOTNET_ACTIONS = {'scanning','ddos','spam','phishing','cryptojacking','credential stuffing', 'ransomware'}

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100
BOTNET_MULTIPLIER = 0.4
FALSE_POSITIVE_WEIGHT = 0.9  # 10% penalty
SCANNER_PENALTY_MULTIPLIER = 0.7  # 30% penalty

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating'] = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)

# ── Raw weighted contributions (no obs additive term) ───────────
df['malicious_scaled'] = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score'] = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']

df['continuity_val'] = df.get('type', '').map(CONTINUITY_MAP).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']

df['tc_raw_rating'] = df['rating'] * Weights['TC_RATING']
df['tc_raw_confidence'] = df['confidence'] * Weights['TC_CONFIDENCE']

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

# ── Observation penalty (multiplier) ────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER = 0.50  # floor so max penalty is 50% reduction (tune as you like)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)

# Apply obs penalty to the base raw score
df['raw_score'] = df['raw_score'] * df['obs_penalty_multiplier']

# ── Botnet flag (robust) ────────────────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    FALSE_POSITIVE_MULTIPLIER = FALSE_POSITIVE_WEIGHT
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0

    # Always show what the FP-applied score would be
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_MULTIPLIER

    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────
df['raw_score'] = np.where(df['botnet_flag'] == 1, df['raw_score'] * BOTNET_MULTIPLIER, df['raw_score'])

# ── Scanner penalty ───────────────────────────────
def has_scanner_tag(tags):
    if isinstance(tags, list):
        return any(str(t).strip().lower() == 'scanner' for t in tags)
    if isinstance(tags, str):
        return 'scanner' in [t.strip().lower() for t in tags.split(',')]
    return False

scanner_mask = df['enrich_tags'].apply(has_scanner_tag)
df.loc[scanner_mask, 'raw_score'] = df.loc[scanner_mask, 'raw_score'] * SCANNER_PENALTY_MULTIPLIER

# ── Absolute Cap (remove obs term since it's a multiplier now) ──
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    max(CONTINUITY_MAP.values()) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 ────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── Assign Severity bin ─────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (now reports obs penalty) ───────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
    }
    total = float(row.get('raw_score', 0) or 0)
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers
    contrib_items = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib_items:
        label = _NAME_MAP.get(k, k)
        if total != 0:
            pct = 100.0 * (v / total)
            driver_bits.append(f"{label} ({v:+.2f}, {pct:+.1f}% of total)")
        else:
            driver_bits.append(f"{label} ({v:+.2f})")

    # Penalty notes
    obs_mult = float(row.get('obs_penalty_multiplier', 1.0) or 1.0)
    obs_note = f"Observations penalty: ×{obs_mult:.2f} ({(1-obs_mult)*100:.1f}% reduction)."

    botnet_flag = int(row.get('botnet_flag', 0) or 0)
    botnet_note = "Botnet penalty applied (60%)." if botnet_flag == 1 else "No botnet penalty."

    fp_cnt = int(row.get('falsePositives', 0) or 0)
    fp_note = f"{fp_cnt} false positive tag(s): 10% penalty applied." if fp_cnt > 0 else "No false positive tags."

    cap_text = f"Raw score uses {100.0 * (total / RAW_SCORE_CAP):.1f}% of cap; normalized to {score:.0f}/1000." if RAW_SCORE_CAP > 0 else f"Normalized score {score:.0f}/1000."

    scanner_penalty = float(row.get('raw_score', 0)) < float(row.get('false_positive_raw_score', 0))
    scanner_note = "Scanner penalty applied (30%)." if scanner_penalty else "No scanner penalty."

    # Add scanner_note to the explanation
    return (
        f"Severity: {sev}. Top drivers: " + "; ".join(driver_bits) + ". "
        f"{obs_note} {botnet_note} {fp_note} {cap_text} {scanner_note}"
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# drop duplicates
df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'malicious_raw_score','continuity_raw_score','rating','tc_raw_rating',
            'tc_raw_confidence','obs_penalty_multiplier','botnet_flag',
            'false_positive_raw_score','falsePositives',
            'raw_score','Threat_Score','Severity','Explanation']])


In [18]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights / Params ────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 2.00,
    # Interpreted as penalty strength in [0,1]; 0.02 ⇒ up to 2% reduction at 365 obs
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
}

BOTNET_ACTIONS = {
    'scanning','ddos','spam','phishing','cryptojacking','credential stuffing','ransomware'
}

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100

# Policy multipliers
FALSE_POSITIVE_WEIGHT = 0.9         # 10% penalty
BOTNET_MULTIPLIER     = 0.4         # 60% penalty when botnet-related
SCANNER_PENALTY_MULTIPLIER = 0.7    # 30% penalty for scanner/benign-crawler tags
DATA_QUALITY_FLOOR    = 0.85        # at worst 15% reduction for poor completeness

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating']     = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)
df['type'] = df.get('type', '').astype(str)

# ── Base additive evidence (no obs additive term) ───────────────
df['malicious_scaled']     = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score']  = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']
df['continuity_val']       = df['type'].map({
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']
df['tc_raw_rating']        = df['rating']     * Weights['TC_RATING']
df['tc_raw_confidence']    = df['confidence'] * Weights['TC_CONFIDENCE']

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

# ── Observation penalty (multiplier; linear) ────────────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER   = 0.50  # floor so max reduction is 50% (tune to taste)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)
df['raw_score'] *= df['obs_penalty_multiplier']

# ── Data quality multiplier (light guard) ───────────────────────
needed = ['type','rating','confidence','enrich_vtMaliciousCount']
present_frac = df[needed].notna().sum(axis=1) / len(needed)
df['data_quality_multiplier'] = present_frac.clip(DATA_QUALITY_FLOOR, 1.0)
df['raw_score'] *= df['data_quality_multiplier']

# ── Botnet flag (robust, multi-word, list/string) ───────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0
    # Preview: always "what if" penalized
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_WEIGHT
    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────────────────────
df['botnet_penalty_multiplier'] = np.where(df['botnet_flag'] == 1, BOTNET_MULTIPLIER, 1.0)
df['raw_score'] *= df['botnet_penalty_multiplier']

# ── Scanner penalty (robust tag parse) ──────────────────────────
def has_scanner_tag(val):
    scanners = {'scanner','masscan','zmap','shodan','censys'}
    if isinstance(val, (list, set, tuple)):
        t = " ".join(map(str, val)).lower()
    elif isinstance(val, str):
        # supports csv-ish strings
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
    return any(s in t for s in scanners)

if 'enrich_tags' in df.columns:
    scanner_mask = df['enrich_tags'].apply(has_scanner_tag)
else:
    scanner_mask = pd.Series(False, index=df.index)

df['scanner_penalty_multiplier'] = np.where(scanner_mask, SCANNER_PENALTY_MULTIPLIER, 1.0)
df['raw_score'] *= df['scanner_penalty_multiplier']

# ── Absolute Cap (multipliers are NOT in cap; only additive parts) ───────────
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    float(df['continuity_val'].max() if len(df) else 3) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 ─────────────────────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── Assign Severity bin ─────────────────────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (drivers + multipliers) ─────────────────────────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
    }
    total = float(row.get('raw_score', 0) or 0)
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers
    contrib = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib:
        label = _NAME_MAP.get(k, k)
        if total != 0:
            pct = 100.0 * (v / total)
            driver_bits.append(f"{label} ({v:+.2f}, {pct:+.1f}% of total)")
        else:
            driver_bits.append(f"{label} ({v:+.2f})")

    # Multipliers
    obs_mult     = float(row.get('obs_penalty_multiplier', 1.0))
    dq_mult      = float(row.get('data_quality_multiplier', 1.0))
    botnet_mult  = float(row.get('botnet_penalty_multiplier', 1.0))
    scanner_mult = float(row.get('scanner_penalty_multiplier', 1.0))
    fp_cnt       = int(row.get('falsePositives', 0) or 0)

    obs_note    = f"Observations ×{obs_mult:.2f} ({(1-obs_mult)*100:.1f}% reduction)."
    dq_note     = f"Data quality ×{dq_mult:.2f}."
    botnet_note = "Botnet ×0.40 applied." if botnet_mult < 1.0 else "No botnet penalty."
    fp_note     = f"{fp_cnt} false positive tag(s): ×0.90 applied." if fp_cnt > 0 else "No false positive tags."
    scan_note   = "Scanner ×0.70 applied." if scanner_mult < 1.0 else "No scanner penalty."

    cap_text = f"Raw score uses {100.0 * (total / RAW_SCORE_CAP):.1f}% of cap; normalized to {score:.0f}/1000." if RAW_SCORE_CAP > 0 else f"Normalized score {score:.0f}/1000."

    return (
        f"Severity: {sev}. Top drivers: " + "; ".join(driver_bits) + ". "
        f"{obs_note} {dq_note} {botnet_note} {fp_note} {scan_note} {cap_text}"
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# ── De-dupe and show ────────────────────────────────────────────
if 'indicator' in df.columns:
    df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'malicious_raw_score','continuity_raw_score','rating','tc_raw_rating',
            'tc_raw_confidence','data_quality_multiplier','obs_penalty_multiplier',
            'botnet_flag','botnet_penalty_multiplier',
            'false_positive_raw_score','falsePositives',
            'scanner_penalty_multiplier',
            'raw_score','Threat_Score','Severity','Explanation']])


,indicator,type,enrich_vtMaliciousCount,obs_count,malicious_raw_score,continuity_raw_score,rating,tc_raw_rating,tc_raw_confidence,data_quality_multiplier,obs_penalty_multiplier,botnet_flag,botnet_penalty_multiplier,false_positive_raw_score,falsePositives,scanner_penalty_multiplier,raw_score,Threat_Score,Severity,Explanation
0,1.192.18.4,Address,4.0,18.0,3.218876,0.1,3.0,0.15,1.32,1.0,0.999014,0,1.0,4.305737,0.0,1.0,4.784153,414,medium,Severity: medium. Top drivers: VT malicious (l...
1,100.27.42.247,Address,0.0,18.0,0.000000,0.1,3.0,0.15,1.56,1.0,0.999014,0,1.0,1.627393,0.0,1.0,1.808215,156,low,Severity: low. Top drivers: TC confidence (+1....
2,101.89.148.7,Address,11.0,45.0,4.969813,0.1,3.0,0.15,1.56,1.0,0.997534,0,1.0,6.086786,0.0,1.0,6.763096,585,high,Severity: high. Top drivers: VT malicious (log...
3,101.89.174.236,Address,8.0,128.0,4.394449,0.1,3.0,0.15,1.34,1.0,0.992986,0,1.0,5.348228,0.0,1.0,5.942476,514,high,Severity: high. Top drivers: VT malicious (log...
4,102.0.5.152,Address,1.0,8.0,1.386294,0.1,4.0,0.20,1.32,1.0,0.999562,1,0.4,2.704479,0.0,1.0,1.201991,104,low,Severity: low. Top drivers: VT malicious (log-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5081,lizearlewellbeing.com,Stripped URL,0.0,10.0,0.000000,0.0,5.0,0.25,1.86,1.0,0.999452,0,1.0,1.897959,0.0,1.0,2.108844,182,low,Severity: low. Top drivers: TC confidence (+1....
5094,prod-barium-us-central-41.li.binaryedge.ninja,Host,0.0,5.0,0.000000,0.2,3.0,0.15,1.56,1.0,0.999726,0,1.0,1.718529,0.0,1.0,1.909477,165,low,Severity: low. Top drivers: TC confidence (+1....
5099,rvthereyet.com,Host,0.0,1.0,0.000000,0.2,4.0,0.20,1.64,1.0,0.999945,1,0.4,1.835899,0.0,1.0,0.815955,71,low,Severity: low. Top drivers: TC confidence (+1....
5102,sepatms.com,Stripped URL,0.0,3.0,0.000000,0.0,5.0,0.25,2.00,1.0,0.999836,0,1.0,2.024667,0.0,1.0,2.249630,195,low,Severity: low. Top drivers: TC confidence (+2....


In [ ]:
# Save the list of indicators from df to the specified CSV file
indicator_list = df['indicator'].drop_duplicates().sort_values()
output_path = r"C:\Users\jaskew\Documents\project_repository\data\indicator library\indicators.csv"
indicator_list.to_csv(output_path, index=False, header=True)
print(f"Saved {len(indicator_list)} indicators to {output_path}")

Saved 1745 indicators to C:\Users\jaskew\Documents\project_repository\data\indicator library\indicators.csv


: 

In [30]:
df[df['indicator'] == '185.156.46.163']

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,continuity_raw_score,tc_raw_rating,tc_raw_confidence,raw_score,obs_penalty_multiplier,botnet_flag,false_positive_raw_score,Threat_Score,Severity,Explanation
1708,68b85370c60b14cf6da420f8,185.156.46.163,2024-06-05T13:07:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-03T08:01:48Z,0,5.0,...,0.1,0.25,1.64,5.547865,0.995397,0,4.993079,476,medium,Severity: medium. Top drivers: VT malicious (l...


In [18]:
df[df['indicator'] == '207.244.71.84']

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,continuity_val,continuity_raw_score,tc_raw_rating,tc_raw_confidence,botnet_flag,raw_score,false_positive_raw_score,Threat_Score,Severity,Explanation
690,68af160dc60b14cf6da41cf9,207.244.71.84,2023-10-23T19:06:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-26T08:06:36Z,0,4.0,...,1.0,0.1,0.2,1.06,0,8.124899,7.312409,603,high,Severity: high. Top drivers: VT malicious (log...


In [32]:
# Calculate the ratio of each Severity bin in df
severity_ratio = (df['Severity'].value_counts(normalize=True) * 100).round(2).astype(str) + '%'
print(severity_ratio)

Severity
medium      54.28%
low         23.61%
high        22.11%
critical      0.0%
Name: proportion, dtype: object
